In [3]:
import arcpy
from arcpy.sa import *
import os
import re

# --- PARAMÈTRES ---
shapefile  = r"C:\Users\admin123\Desktop\viirs_data\arcgis\shape_file\Provinces_and_Territories_of_Canada_mollweide.shp"
zone_field = "Nom_FR"
viirs_dir  = r"C:\Users\admin123\Desktop\viirs_data\viirs_image\resampled"
out_base   = r"C:\Users\admin123\Desktop\viirs_data\arcgis\resultat"
data_base  = r"C:\Users\admin123\Desktop\viirs_data\arcgis\data_base"

# --- SETUP ---
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True
arcpy.env.workspace = viirs_dir

for d in [out_base, data_base]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- LECTURE DES COORDONNÉES ET AREA DU SHAPEFILE ---
coord_dict = {}
area_dict  = {}

with arcpy.da.SearchCursor(shapefile, [zone_field, "Y", "X", "AREA"]) as cur:
    for row in cur:
        coord_dict[row[0]] = (row[1], row[2])
        area_dict[row[0]]  = row[3]

print(f"Coordonnées et surfaces chargées pour {len(coord_dict)} zones : {list(coord_dict.keys())}")

# --- TRAITEMENT RASTER PAR RASTER ---
rasters = arcpy.ListRasters("*", "TIF")
if not rasters:
    raise Exception("Aucun raster .tif trouvé dans viirs_dir.")

tables_creees = []
for ras in rasters:
    match = re.search(r"20\d{2}", ras)
    if not match:
        print(f"⚠️  Année non trouvée dans : {ras} -> ignoré")
        continue
    year = match.group(0)
    print(f"Traitement : {ras} (année {year})")

    # 1. Supprimer les valeurs négatives (-> NoData)
    raster_obj     = Raster(os.path.join(viirs_dir, ras))
    raster_cleaned = SetNull(raster_obj < 0, raster_obj)

    # 2. Zonal statistics sur le raster nettoyé
    out_table = os.path.join(out_base, f"viirs_{year}.dbf")
    ZonalStatisticsAsTable(
        in_zone_data    = shapefile,
        zone_field      = zone_field,
        in_value_raster = raster_cleaned,
        out_table       = out_table,
        ignore_nodata   = "DATA",
        statistics_type = "ALL"
    )

    # 3. Ajout du champ Annee
    existing_fields = [f.name for f in arcpy.ListFields(out_table)]
    if "Annee" not in existing_fields:
        arcpy.AddField_management(out_table, "Annee", "SHORT")
    with arcpy.da.UpdateCursor(out_table, ["Annee"]) as cur:
        for row in cur:
            row[0] = int(year)
            cur.updateRow(row)

    # 4. Ajout des champs Latitude, Longitude et AREA
    existing_fields = [f.name for f in arcpy.ListFields(out_table)]
    for field_name, field_type in [("Latitude",  "DOUBLE"),
                                    ("Longitude", "DOUBLE"),
                                    ("AREA",      "DOUBLE")]:
        if field_name not in existing_fields:
            arcpy.AddField_management(out_table, field_name, field_type)

    with arcpy.da.UpdateCursor(out_table, [zone_field, "Latitude", "Longitude", "AREA"]) as cur:
        for row in cur:
            nom = row[0]
            if nom in coord_dict:
                row[1] = coord_dict[nom][0]   # Latitude  (Y du shapefile)
                row[2] = coord_dict[nom][1]   # Longitude (X du shapefile)
                row[3] = area_dict[nom]        # AREA
                cur.updateRow(row)
            else:
                print(f"  ⚠️  Zone '{nom}' absente du shapefile, champs non remplis")

    tables_creees.append(out_table)
    print(f"  ✅ Table créée : {out_table}")

print("Zonal statistics terminé pour toutes les années trouvées.")

# --- FUSION ---
if tables_creees:
    merged_table = os.path.join(data_base, "viirs_zonal_2012_2024_moll_resamp.dbf")
    arcpy.Merge_management(tables_creees, merged_table)
    print(f"Table fusionnée créée : {merged_table}")
else:
    print("Aucune table créée, rien à fusionner.")
    merged_table = None

# --- EXPORT CSV ---
if merged_table and arcpy.Exists(merged_table):
    try:
        arcpy.conversion.TableToTable(
            in_rows  = merged_table,
            out_path = out_base,
            out_name = "viirs_zonal_2012_2024_moll_resampl.csv"
        )
        print(f"✅ Export CSV terminé : {os.path.join(out_base, 'viirs_zonal_2012_2024_moll_resampl.csv')}")
    except arcpy.ExecuteError:
        print("❌ Erreur lors de l'export CSV :")
        print(arcpy.GetMessages(2))
    except Exception as e:
        print(f"❌ Erreur inattendue export CSV : {e}")

# --- SUPPRESSION DES TABLES INTERMÉDIAIRES ---
for t in tables_creees:
    if arcpy.Exists(t):
        arcpy.Delete_management(t)
        print(f"🗑️  Supprimé : {t}")
print("✅ Toutes les tables intermédiaires ont été supprimées.")
print("✅ Script terminé.")

Coordonnées et surfaces chargées pour 13 zones : ['Manitoba', 'Saskatchewan', 'Alberta', 'Colombie-Britannique', 'Nunavut', 'Territoires du Nord-Ouest', 'Yukon', 'Ontario', 'Québec', 'Nouveau-Brunswick', 'Nouvelle-Écosse', 'Terre-Neuve-et-Labrador', 'Île du Prince-Édouard']
Traitement : VNL_npp_2023_global_vcmslcfg_v2_c202402081600.average_masked_mollweide_1km.tif (année 2023)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\viirs_2023.dbf
Traitement : VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked_mollweide_1km.tif (année 2024)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\viirs_2024.dbf
Traitement : VNL_v21_npp_2012_global_vcmcfg_c202205302300.average_masked_mollweide_1km.tif (année 2012)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs_data\arcgis\resultat\viirs_2012.dbf
Traitement : VNL_v21_npp_2013_global_vcmcfg_c202205302300.average_masked_mollweide_1km.tif (année 2013)
  ✅ Table créée : C:\Users\admin123\Desktop\viirs